# Übung 4: Arbeiten mit Zeitreihen in der Energiewirtschaft

**Ziel:** CSV-Daten einlesen, Zeitreihen verarbeiten, und energiewirtschaftliche
Berechnungen durchführen.

## 4.1 Daten vorbereiten

Wir arbeiten mit zwei Datensätzen:
- **Haushaltslastprofil** (stündlich, kW)
- **Strompreise** (stündlich, EUR/MWh)

Zunächst erzeugen wir synthetische Beispieldaten (in der Praxis würdet ihr echte CSVs einlesen).

In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

idx = pd.date_range("2024-01-01", "2024-12-31 23:00", freq="h")
stunde = idx.hour.to_numpy()
monat = idx.month.to_numpy()

# Haushaltslast (kW): Tagesprofil + Winter-Aufschlag + Rauschen
tagesprofil = np.maximum(0, np.sin((stunde - 6) * np.pi / 12))
winter = 0.3 * (monat <= 3)
last_kw = 0.4 + 0.6 * tagesprofil + winter + np.random.normal(0, 0.1, len(idx))
last_kw = np.maximum(0.1, last_kw)

# Strompreise (EUR/MWh): Saison- + Tagesprofil + Rauschen
preise = (70 + 20 * np.sin((monat - 1) * np.pi / 6)
            + 15 * np.sin((stunde - 6) * np.pi / 12)
            + np.random.normal(0, 12, len(idx)))
preise[np.random.choice(len(idx), 150, replace=False)] = np.random.uniform(-20, -1, 150)

df_last = pd.DataFrame({"last_kw": last_kw}, index=idx)
df_preis = pd.DataFrame({"price_eur_mwh": preise}, index=idx)
df_last.to_csv("haushaltslast_2024.csv")
df_preis.to_csv("strompreise_2024.csv")

f"Daten erzeugt: {len(idx)} Stunden von {idx[0]} bis {idx[-1]}" 

'Daten erzeugt: 8784 Stunden von 2024-01-01 00:00:00 bis 2024-12-31 23:00:00'

## 4.2 CSV-Dateien einlesen

In [2]:
df_last = pd.read_csv("haushaltslast_2024.csv", index_col=0, parse_dates=True)
df_last.head()

,last_kw
2024-01-01 00:00:00,0.749671
2024-01-01 01:00:00,0.686174
2024-01-01 02:00:00,0.764769
2024-01-01 03:00:00,0.852303
2024-01-01 04:00:00,0.676585


In [3]:
df_preis = pd.read_csv("strompreise_2024.csv", index_col=0, parse_dates=True)
df_preis.head()

,price_eur_mwh
2024-01-01 00:00:00,40.553036
2024-01-01 01:00:00,58.214193
2024-01-01 02:00:00,64.561947
2024-01-01 03:00:00,55.804569
2024-01-01 04:00:00,70.651526


In [4]:
df = pd.merge(df_last, df_preis, left_index=True, right_index=True)
df.head()

,last_kw,price_eur_mwh
2024-01-01 00:00:00,0.749671,40.553036
2024-01-01 01:00:00,0.686174,58.214193
2024-01-01 02:00:00,0.764769,64.561947
2024-01-01 03:00:00,0.852303,55.804569
2024-01-01 04:00:00,0.676585,70.651526


## 4.3 Grundlegende Statistiken

In [5]:
df.describe().round(2)

,last_kw,price_eur_mwh
count,8784.00,8784.00
mean,0.66,68.67
std,0.29,23.51
min,0.10,-19.89
25%,0.42,54.16
50%,0.63,69.33
75%,0.88,84.80
max,1.59,143.52


In [6]:
jahresverbrauch_kwh = df["last_kw"].sum()       # kW × 1h = kWh
spitzenlast_kw = df["last_kw"].max()
mittlere_last_kw = df["last_kw"].mean()

pd.Series({
    "Jahresverbrauch [kWh]":        jahresverbrauch_kwh,
    "Spitzenlast [kW]":             spitzenlast_kw,
    "Mittlere Last [kW]":           mittlere_last_kw,
    "Benutzungsdauer [h/a]":        jahresverbrauch_kwh / spitzenlast_kw,
    "Mittlerer Preis [EUR/MWh]":    df["price_eur_mwh"].mean(),
    "Negative Preisstunden [h/a]":  (df["price_eur_mwh"] < 0).sum(),
}).round(2)

Jahresverbrauch [kWh]          5836.90
Spitzenlast [kW]                  1.59
Mittlere Last [kW]                0.66
Benutzungsdauer [h/a]          3663.34
Mittlerer Preis [EUR/MWh]        68.67
Negative Preisstunden [h/a]     150.00
dtype: float64

## 4.4 Stromkosten berechnen

In [7]:
# Kosten pro Stunde: Last (kW) × Preis (EUR/MWh) × 1h / 1000 (kW→MW)
df["kosten_eur"] = df["last_kw"] * df["price_eur_mwh"] / 1000
df[["last_kw", "price_eur_mwh", "kosten_eur"]].head(10).round(4)

,last_kw,price_eur_mwh,kosten_eur
2024-01-01 00:00:00,0.7497,40.5530,0.0304
2024-01-01 01:00:00,0.6862,58.2142,0.0399
2024-01-01 02:00:00,0.7648,64.5619,0.0494
2024-01-01 03:00:00,0.8523,55.8046,0.0476
2024-01-01 04:00:00,0.6766,70.6515,0.0478
2024-01-01 05:00:00,0.6766,57.5961,0.0390
2024-01-01 06:00:00,0.8579,44.6237,0.0383
2024-01-01 07:00:00,0.9320,68.7955,0.0641
2024-01-01 08:00:00,0.9531,78.2588,0.0746
2024-01-01 09:00:00,1.1785,77.2965,0.0911


In [8]:
jahreskosten = df["kosten_eur"].sum()
durchschnittspreis = jahreskosten / (jahresverbrauch_kwh / 1000)   # EUR/MWh

pd.Series({
    "Jahreskosten [EUR]":           jahreskosten,
    "Durchschnittspreis [EUR/MWh]": durchschnittspreis,
    "Spezifische Kosten [ct/kWh]":  durchschnittspreis / 10,
}).round(2)

Jahreskosten [EUR]              426.15
Durchschnittspreis [EUR/MWh]     73.01
Spezifische Kosten [ct/kWh]       7.30
dtype: float64

## 4.5 Zeitliche Aggregation (Resample)

In [9]:
df_monat = df.resample("ME").agg(
    verbrauch_kwh=("last_kw", "sum"),
    spitzenlast_kw=("last_kw", "max"),
    mittlerer_preis=("price_eur_mwh", "mean"),
    monatskosten_eur=("kosten_eur", "sum"),
)
df_monat.round(2)

,verbrauch_kwh,spitzenlast_kw,mittlerer_preis,monatskosten_eur
2024-01-31,661.02,1.55,69.17,47.52
2024-02-29,626.17,1.54,77.98,50.51
2024-03-31,664.33,1.59,83.52,57.11
2024-04-30,425.85,1.22,88.03,39.23
2024-05-31,438.92,1.22,85.81,39.36
2024-06-30,419.65,1.22,78.89,34.50
2024-07-31,438.20,1.24,69.32,32.11
2024-08-31,434.47,1.31,59.20,27.36
2024-09-30,423.52,1.26,52.30,23.74
2024-10-31,436.10,1.22,49.05,22.87


In [10]:
df_tag = df.resample("D").agg(
    mittlere_last_kw=("last_kw", "mean"),
    spitzenlast_kw=("last_kw", "max"),
    mittlerer_preis=("price_eur_mwh", "mean"),
    tageskosten_eur=("kosten_eur", "sum"),
)
df_tag.head(10).round(2)

,mittlere_last_kw,spitzenlast_kw,mittlerer_preis,tageskosten_eur
2024-01-01,0.88,1.32,66.54,1.44
2024-01-02,0.86,1.32,70.92,1.53
2024-01-03,0.90,1.38,68.60,1.54
2024-01-04,0.88,1.37,66.37,1.47
2024-01-05,0.89,1.41,69.47,1.54
2024-01-06,0.88,1.33,68.50,1.52
2024-01-07,0.91,1.49,71.18,1.64
2024-01-08,0.89,1.55,70.75,1.57
2024-01-09,0.92,1.38,73.58,1.68
2024-01-10,0.88,1.35,68.24,1.48


## 4.6 Stündliches Profil (Tagesdurchschnitt)

In [11]:
profil_stunde = df.groupby(df.index.hour).agg(
    last_kw=("last_kw", "mean"),
    price_eur_mwh=("price_eur_mwh", "mean"),
)
profil_stunde.round(2)

,last_kw,price_eur_mwh
0,0.48,54.09
1,0.47,55.44
2,0.47,55.33
3,0.47,58.64
4,0.47,60.79
5,0.48,66.08
6,0.47,68.74
7,0.63,73.20
8,0.78,75.85
9,0.89,79.74


In [12]:
df["werktag"] = df.index.weekday < 5
df["stunde"] = df.index.hour

profil = df.groupby(["werktag", "stunde"])["last_kw"].mean()
pd.DataFrame({
    "Werktag (kW)":    profil.loc[True],
    "Wochenende (kW)": profil.loc[False],
}).round(3)

,Werktag (kW),Wochenende (kW)
stunde,,
0,0.474,0.478
1,0.469,0.461
2,0.469,0.490
3,0.470,0.477
4,0.471,0.474
5,0.480,0.486
6,0.468,0.471
7,0.639,0.622
8,0.778,0.767


## 4.7 Ausblick: Leistungspreis-Berechnung (90%-Peak)

In der Praxis zahlen Industriekunden einen **Leistungspreis** basierend auf
ihrer Spitzenlast. Oft wird diese über die höchsten 15-Minuten-Werte bestimmt.

**Vereinfachte Variante:** Wir berechnen den Leistungswert als den Schwellenwert,
unterhalb dessen 90% aller Stundenwerte liegen (90%-Quantil).

In [13]:
quantil_90 = df["last_kw"].quantile(0.9)
stunden_ueber_90 = (df["last_kw"] > quantil_90).sum()
leistungspreis = 120   # EUR/kW/a
leistungskosten = quantil_90 * leistungspreis

pd.Series({
    "90%-Quantil der Last [kW]":          quantil_90,
    "Absolute Spitzenlast [kW]":          df["last_kw"].max(),
    "Stunden über 90%-Schwelle [h]":      stunden_ueber_90,
    "Leistungskosten [EUR/a]":            leistungskosten,
    "Gesamtkosten (Arb.+Leist.) [EUR/a]": jahreskosten + leistungskosten,
}).round(2)

90%-Quantil der Last [kW]               1.07
Absolute Spitzenlast [kW]               1.59
Stunden über 90%-Schwelle [h]         879.00
Leistungskosten [EUR/a]               128.01
Gesamtkosten (Arb.+Leist.) [EUR/a]    554.16
dtype: float64

## 4.8 Übungsaufgaben

Jetzt arbeiten wir mit echten Zeitreihen aus diesem Ordner:
- **`dayahead_2025.csv`** — Day-Ahead-Börsenpreise Deutschland 2025 (15 min, UTC, Spalte `dayahead €/MWh`)
- **`CHR11_last_15min_2025.csv`** — Lastprofil einer Studentin (15 min, UTC, Spalte `last [kW]`)

*Einlesen:* `pd.read_csv("<datei>.csv", parse_dates=["timestamp"], index_col="timestamp")`

### Aufgabe 1: Day-Ahead-Preise untersuchen

Lies `dayahead_2025.csv` ein und bestimme:
- den **höchsten** und **niedrigsten** Preis (inkl. Zeitpunkt)
- den **Durchschnittspreis** und die **Standardabweichung**
- die Anzahl der **15-min-Intervalle mit negativen Preisen**

In [14]:
# Dein Code hier:


### Aufgabe 2: Lastprofil untersuchen

Lies `CHR11_last_15min_2025.csv` ein und bestimme:
- die **Spitzenlast** und die **minimale Leistung** (inkl. Zeitpunkt)
- die **mittlere Leistung**
- den **Jahresverbrauch** in kWh

*Tipp:* Energie pro Intervall = Leistung [kW] × Intervalldauer [h]. Die Intervalldauer hier: 0.25 h.

In [15]:
# Dein Code hier:


### Aufgabe 3: Fixtarif vs. dynamischer Tarif

Die Studentin überlegt, auf einen dynamischen Stromtarif (15-min-Preis aus dem Day-Ahead-Markt) zu wechseln. Vergleiche die Jahreskosten mit einem klassischen **Fixtarif**.

**Annahme:** Auf den Börsenpreis kommen pauschal **+20 ct/kWh** Aufschlag (Netzentgelte, Umlagen, Steuern, Vertriebsmarge). Der Aufschlag ist in beiden Tarifen gleich:

$$p_\text{Haushalt}(t) = p_\text{Börse}(t) + 20\ \text{ct/kWh}$$

1. Führe Last und Börsenpreis über den gemeinsamen Zeitindex zusammen (`df_last.join(df_preis, how="inner")`).
2. **Fixtarif** — Jahresmittelwert des Haushaltspreises × Jahresverbrauch.
3. **Dynamischer Tarif** — pro 15-min-Intervall `Energie[kWh] × Haushaltspreis[ct/kWh]`, aufsummiert.
4. Gib den **Preisunterschied** in € und in % aus. Welcher Tarif ist günstiger?

*Einheitenumrechnung:* 1 €/MWh = 0.1 ct/kWh (also 20 ct/kWh = 200 €/MWh).

In [16]:
# Dein Code hier:
